# Sample statistics for the paper (sample description; PMs within Gaia's magnitude range; Gaia sources without a fit)

1. Imaging: date range and measurements per band (measured epochs of Gaia stars, which sample every exposure);
   total measurements used in the PM fits (sum of `pm_n_epochs`).
2. Dustin: does the catalogue add PMs *within* Gaia's magnitude range? Clean PMs for point sources with no Gaia PM
   (not in Gaia, or Gaia two-parameter), by r, compared with Gaia sources that have a PM; and how many of them move
   significantly (|mu|/sigma > 5), a sign that they are stars.
3. Paper \todo: why ~4% of Gaia point sources (with a Gaia PM) have no fit, after removing brick-overlap duplicates.

Numbers only (no paper figure). Combined v2 files, all slices, 16 workers.

In [1]:
import os, sys
import numpy as np
import fitsio
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
sys.path.append(os.path.abspath('..'))
from py_files.pm_analysis import COMBINED_DIR
DEGREES = [f'{d:03d}' for d in range(64)]
RB = np.arange(14, 26.01, 0.5)

def mag(f):
    with np.errstate(all='ignore'):
        return 22.5 - 2.5 * np.log10(f)

def per_slice(d):
    fp, fg, fe, fb = (os.path.join(COMBINED_DIR, f'{k}-{d}.fits') for k in ('pm', 'gaia-objects', 'gaia-epochs', 'bricks'))
    if not os.path.exists(fp):
        return None
    out = {}
    E = fitsio.read(fe, columns=['mjd', 'filter', 'full_fit_dra_ivar', 'full_fit_dra'])
    E = E[(E['full_fit_dra_ivar'] > 0) & (E['full_fit_dra'] != 0)]
    out['mjd_min'], out['mjd_max'] = float(E['mjd'].min()), float(E['mjd'].max())
    band = np.char.strip(E['filter'].astype(str))
    out['gaia_epochs_band'] = np.array([np.sum(band == b) for b in 'griz'])
    out['gaia_epochs_year'] = np.histogram(2000 + (E['mjd'] - 51544.5) / 365.25, np.arange(2012, 2024))[0]
    P = fitsio.read(fp, columns=['flux_r', 'ref_cat', 'pmra_ivar', 'pm_quality', 'pm_n_epochs', 'pm_baseline_days',
                                 'pmra_no_plx', 'pmdec_no_plx', 'pmra_no_plx_ivar', 'pmdec_no_plx_ivar'])
    clean = P['pm_quality'] == 0
    out['n_meas_all'] = int(P['pm_n_epochs'].astype(np.int64).sum())
    out['n_meas_clean'] = int(P['pm_n_epochs'][clean].astype(np.int64).sum())
    out['n_obj_all'], out['n_obj_clean'] = len(P), int(clean.sum())
    r = mag(P['flux_r'])
    with np.errstate(all='ignore'):
        s = np.sqrt((1 / P['pmra_no_plx_ivar'] + 1 / P['pmdec_no_plx_ivar']) / 2)
        snr = np.hypot(P['pmra_no_plx'] * np.sqrt(P['pmra_no_plx_ivar']), P['pmdec_no_plx'] * np.sqrt(P['pmdec_no_plx_ivar'])) / np.sqrt(2)
    gaia = np.char.strip(P['ref_cat'].astype(str)) == 'GE'
    gpm = gaia & (P['pmra_ivar'] > 0)
    h = lambda sel: np.histogram(r[sel], RB)[0]
    for name, sel in [('gaia_pm', clean & gpm), ('nogaiapm', clean & ~gpm), ('nogaiapm_s5', clean & ~gpm & (s < 5)),
                      ('nongaia_s5', clean & ~gaia & (s < 5)), ('gaia2p_s5', clean & gaia & ~gpm & (s < 5)),
                      ('nogaiapm_s5_moving', clean & ~gpm & (s < 5) & (snr > 5)), ('gaia_pm_moving', clean & gpm & (snr > 5))]:
        out['h_' + name] = h(sel)
    G = fitsio.read(fg, columns=['ref_id', 'type', 'pm_flag', 'pm_n_epochs', 'pmra_ivar', 'ra', 'dec', 'brickname'])
    out['g_ref_id'] = G['ref_id'].astype(np.int64)
    out['g_gpm'] = G['pmra_ivar'] > 0
    out['g_psf'] = np.char.strip(G['type'].astype(str)) == 'PSF'
    out['g_fit'] = G['pm_flag'] == 1
    out['g_nep'] = G['pm_n_epochs'].astype(np.int32)
    out['g_ra'], out['g_dec'] = G['ra'], G['dec']
    out['bricks'] = np.char.strip(fitsio.read(fb, columns=['brickname'])['brickname'].astype(str))
    return out

with ProcessPoolExecutor(16, mp_context=mp.get_context('fork')) as ex:
    parts = [p for p in ex.map(per_slice, DEGREES) if p is not None]
C = {}
for k in parts[0]:
    if k == 'mjd_min': C[k] = min(p[k] for p in parts)
    elif k == 'mjd_max': C[k] = max(p[k] for p in parts)
    elif k.startswith('g_') or k == 'bricks': C[k] = np.concatenate([p[k] for p in parts])
    else: C[k] = sum(p[k] for p in parts)
print(len(parts), 'slices')

64 slices


In [2]:
# 1. imaging
from astropy.time import Time
t0, t1 = Time(C['mjd_min'], format='mjd').iso[:10], Time(C['mjd_max'], format='mjd').iso[:10]
print(f'measured epochs of Gaia stars span {t0} to {t1} (MJD {C["mjd_min"]:.0f}-{C["mjd_max"]:.0f})')
gb = C['gaia_epochs_band']
print('measurements of Gaia stars by band: ' + ', '.join(f'{b} {n/gb.sum():.1%}' for b, n in zip('griz', gb)) + f'  (total {gb.sum():,})')
print('by year:', dict(zip(range(2012, 2023), (C['gaia_epochs_year'] / C['gaia_epochs_year'].sum()).round(3))))
print(f"measurements used in PM fits: all {C['n_meas_all']/1e9:.2f} billion ({C['n_obj_all']/1e6:.1f}M objects), "
      f"clean {C['n_meas_clean']/1e9:.2f} billion ({C['n_obj_clean']/1e6:.1f}M objects)")

measured epochs of Gaia stars span 2013-09-01 to 2021-08-30 (MJD 56536-59456)
measurements of Gaia stars by band: g 25.6%, r 28.3%, i 21.4%, z 24.7%  (total 507,703,722)
by year: {2012: 0.0, 2013: 0.053, 2014: 0.122, 2015: 0.117, 2016: 0.163, 2017: 0.162, 2018: 0.223, 2019: 0.092, 2020: 0.019, 2021: 0.048, 2022: 0.0}
measurements used in PM fits: all 8.08 billion (266.0M objects), clean 6.63 billion (202.2M objects)


In [3]:
# 2. PMs within Gaia's magnitude range: clean, sigma < 5 mas/yr, objects with no Gaia PM, vs clean Gaia stars with a PM
rc = 0.5 * (RB[1:] + RB[:-1])
print(f"{'r':>5s} {'Gaia w/ PM':>11s} {'no Gaia PM':>11s} {'(non-Gaia':>10s} {'Gaia 2p)':>9s} {'extra':>6s} {'moving extra':>13s} {'moving Gaia':>12s}")
for i in range(len(rc)):
    if rc[i] < 15.5 or rc[i] > 22: continue
    g, x = C['h_gaia_pm'][i], C['h_nogaiapm_s5'][i]
    print(f"{rc[i]:5.2f} {g:11,d} {x:11,d} {C['h_nongaia_s5'][i]:10,d} {C['h_gaia2p_s5'][i]:9,d} {x / max(g, 1):6.1%} "
          f"{C['h_nogaiapm_s5_moving'][i] / max(x, 1):13.1%} {C['h_gaia_pm_moving'][i] / max(g, 1):12.1%}")
for hi in (20, 21):
    m = rc < hi
    g, x = C['h_gaia_pm'][m].sum(), C['h_nogaiapm_s5'][m].sum()
    print(f'r < {hi}: clean Gaia stars with a PM {g:,}; clean PMs (sigma < 5) without a Gaia PM {x:,} (+{x/g:.1%}): '
          f"non-Gaia {C['h_nongaia_s5'][m].sum():,}, Gaia 2-parameter {C['h_gaia2p_s5'][m].sum():,}; "
          f"moving at > 5 sigma: {C['h_nogaiapm_s5_moving'][m].sum() / x:.1%} (Gaia stars: {C['h_gaia_pm_moving'][m].sum() / g:.1%})")

    r  Gaia w/ PM  no Gaia PM  (non-Gaia  Gaia 2p)  extra  moving extra  moving Gaia
15.75      37,193         439          8       431   1.2%         40.5%        57.0%
16.25     284,498       1,263         35     1,228   0.4%         49.8%        58.4%
16.75     414,510       2,320         50     2,270   0.6%         51.2%        58.9%
17.25     560,747       3,696         69     3,627   0.7%         53.7%        58.9%
17.75     708,573       5,412        216     5,196   0.8%         54.5%        58.0%
18.25     866,928       6,724        458     6,266   0.8%         50.7%        56.1%
18.75   1,084,691       7,569        728     6,841   0.7%         46.0%        51.3%
19.25   1,352,298       8,885      1,154     7,731   0.7%         41.3%        45.6%
19.75   1,526,941      11,148      1,665     9,483   0.7%         34.8%        42.1%
20.25   1,856,181      20,663      2,599    18,064   1.1%         27.2%        33.3%
20.75   1,602,497     618,353     85,520   532,833  38.6%        

In [4]:
# 3. Gaia point sources with a Gaia PM but no fit, after de-duplication by ref_id (prefer fitted rows)
order = np.lexsort((~C['g_psf'], ~C['g_fit'], C['g_ref_id']))
first = order[np.r_[True, C['g_ref_id'][order][1:] != C['g_ref_id'][order][:-1]]]
U = {k[2:]: C[k][first] for k in C if k.startswith('g_')}
sel = U['gpm'] & U['psf'] & ~U['fit']
n_all = (U['gpm'] & U['psf']).sum()
nep = U['nep'][sel]
print(f'{sel.sum():,} of {n_all:,} Gaia point sources with a Gaia PM have no fit ({sel.sum()/n_all:.1%}): '
      f'0 measured epochs {np.mean(nep == 0):.1%}, 1-2 {np.mean((nep > 0) & (nep < 3)):.1%}, >= 3 {np.mean(nep >= 3):.1%}')
# for those with no measured epochs: which DR10 brick contains them, and is it one we processed?
SB = fitsio.read('/global/cfs/cdirs/cosmo/data/legacysurvey/dr10/survey-bricks.fits.gz', columns=['BRICKNAME', 'RA1', 'RA2', 'DEC1', 'DEC2'])
done = set(C['bricks'].tolist())
z = np.flatnonzero(sel)[nep == 0]
rng = np.random.default_rng(0)
z = rng.choice(z, min(20000, len(z)), replace=False)
o = np.argsort(SB['DEC1'])
dec1 = SB['DEC1'][o]
inproc = []
for ra, dec in zip(U['ra'][z], U['dec'][z]):
    lo, hi = np.searchsorted(dec1, dec - 0.26), np.searchsorted(dec1, dec + 0.01)
    c = o[lo:hi]
    c = c[(SB['DEC1'][c] <= dec) & (SB['DEC2'][c] > dec) & (SB['RA1'][c] <= ra) & (SB['RA2'][c] > ra)]
    inproc.append(str(SB['BRICKNAME'][c[0]]).strip() in done if len(c) else False)
inproc = np.array(inproc)
print(f'of those with no measured epochs ({len(z):,} sampled): containing brick not processed {np.mean(~inproc):.1%}, '
      f'processed {np.mean(inproc):.1%}')

769,949 of 18,489,181 Gaia point sources with a Gaia PM have no fit (4.2%): 0 measured epochs 8.9%, 1-2 36.0%, >= 3 55.1%


of those with no measured epochs (20,000 sampled): containing brick not processed 16.9%, processed 83.1%
